In [ ]:
from agents import Agent
from pydantic import BaseModel, Field

from dotenv import load_dotenv


### Clarifying Agent

Ask clarifying questions associated with the query.

In [ ]:
CLARIFYING_AGENT_MODEL = "gpt-4o-mini"

class ClarifyingQuestions(BaseModel):
    questions: list[str] = Field(
        description="Exactly 3 clarifying questions to better understand the user's research needs.",
        min_items=3,  
        max_items=3   
    )

CLARIFYING_AGENT_INSTRUCTIONS = "You are a helpful assistant that anayses the user's query and asks clarifying questions to help the user refine their query.\
    Analyse the user's query and ask them 3 questions to help them refine their query. \
    Once you get these questions, please generate a comprehensive research query that will be used to search the web for relevant information."

clarifying_agent = Agent(
    name="Clarifying Agent",
    instructions=CLARIFYING_AGENT_INSTRUCTIONS,
    model=CLARIFYING_AGENT_MODEL,
    output_type=ClarifyingQuestions
)

### Evaluator Agent

Evaluates whether the answer generated so far for our query is good enough and meets the user's needs.

In [ ]:
class Evaluation(BaseModel):
    is_acceptable: bool
    feedback: str

EVALUATOR_AGENT_MODEL = "gpt-4o-mini"
EVALUATOR_AGENT_INSTRUCTIONS_PLACEHOLDER = "You are an evaluator that decides whether a response to a question is acceptable. \
You are provided with a conversation between a User and an Agent. Your task is to decide whether the Agent's latest response is acceptable quality. \
The Agent is playing the role of {name} and is representing {name} on their website. \
The Agent has been instructed to be professional and engaging, as if talking to a potential client or future employer who came across the website. \
The Agent has been provided with context on {name} in the form of their summary and LinkedIn details. Here's the information:"

EVALUATOR_AGENT_INSTRUCTIONS = ""

evaluator_agent = Agent(
    name="Evaluator Agent",
    instructions=EVALUATOR_AGENT_INSTRUCTIONS,
    model=EVALUATOR_AGENT_MODEL,
    output_type=Evaluation)

### Research Manager Agent
It should be autonomous with the ability to decide whether it needs to do more research.
- agents as tools
- hand-offs

In [ ]:
RESEARCH_MANAGER_INSTRUCTIONS = """You are a research manager that oversees the research process of the query sent by the user. 
You have access to a team of agents that can help you with the research. 
You are responsible for deciding whether the research is complete and the answer is good enough. 
You can use the agents as tools to help you with the research. 

Please follow the following steps carefully for a successful and efficient research process:
1. Ask the clarifying agent to ask questions to the user to better understand and refine their research needs.
2. Use the query formulated by the clarifying agent to search the web for relevant information.
3. Evaluation: Use the evaluator agent to evaluate the research.
4. Iteration: If the research is not good enough, repeat the process.
5. Answer: Once the research is complete and the answer is good enough, return the answer to the user.
"""

RESEARCH_MANAGER_MODEL = "gpt-4o-mini"

research_manager = Agent(
    name="Research Manager",
    instructions=RESEARCH_MANAGER_INSTRUCTIONS,
    model=RESEARCH_MANAGER_MODEL)